# 2. Turn prices into a report you understand

A desk report should make sense **before** we automate it. You will calculate and explain returns, rolling volatility, and drawdown, then read a four-panel report.
The same notebook can later be run for another stock by changing its parameters.

**By the end:** explain an unfamiliar output row, change a parameter intentionally, and check a calculation on tiny data.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sqlite3
from pathlib import Path
from IPython.display import display
%matplotlib inline
course_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').is_file())
lesson_dir = course_root / 'Lectures/Lecture 4'
plt.style.use('seaborn-v0_8-whitegrid')
import scrapbook as sb

## 1. Parameters are ordinary Python variables
These values define the report's question. Papermill will later override this tagged cell; for now, edit it like any Python cell.
Use the bCourses database. It contains historical observations; this is not a live market report.

In [ ]:
ticker = 'AAPL'
start_date = '2024-01-01'
end_date = '2024-06-30'
db_path = str(lesson_dir / 'data/data.db')
volatility_window = 20

### Load the data
This is the SQL pattern from the previous module, plus useful messages for a missing file, reversed dates, or insufficient history.
Compare the displayed date range and first rows with your report parameters.

In [ ]:
database = Path(db_path).expanduser().resolve()
if not database.is_file():
    raise FileNotFoundError(f'Download the bCourses database to {database}')
if pd.Timestamp(start_date) > pd.Timestamp(end_date):
    raise ValueError('start_date must be on or before end_date.')
if not isinstance(volatility_window, int) or isinstance(volatility_window, bool) or volatility_window < 2:
    raise ValueError('volatility_window must be an integer >= 2.')
query = """
SELECT ts, open, high, low, close, adj_close, volume
FROM ohlc WHERE ticker = ? AND ts >= ? AND ts <= ? ORDER BY ts
"""
from contextlib import closing
with closing(sqlite3.connect(database.as_uri() + '?mode=ro', uri=True)) as conn:
    df = pd.read_sql_query(query, conn, params=(ticker, start_date, end_date), parse_dates=['ts'], index_col='ts')
if len(df) < volatility_window + 1:
    raise ValueError(f'{ticker}: need at least {volatility_window + 1} prices for a full return window; got {len(df)}.')
print(f'Loaded {len(df)} trading days for {ticker}: {df.index.min().date()} to {df.index.max().date()}')
display(df.head())

### Decide whether the data can support the report
Our supplied checker flags missing/non-finite values, nonpositive equity prices, impossible OHLC bounds, negative volume, and duplicate dates.
Open/high/low/close are raw prices; adjusted close follows the vendor's adjustment convention and is checked separately for positivity.
**Predict:** would checking only `close > 0` catch a missing volume or a duplicated date?

In [ ]:
def quality_flags(frame):
    """Flag missing/non-finite values, nonpositive equity prices, bad OHLC, and duplicate dates."""
    price_cols = ['open', 'high', 'low', 'close', 'adj_close']
    flags = pd.DataFrame(index=frame.index)
    flags['missing_or_nonfinite'] = ~np.isfinite(frame[price_cols + ['volume']]).all(axis=1)
    flags['nonpositive_price'] = frame[price_cols].le(0).any(axis=1)
    flags['ohlc_bounds'] = (frame['high'].lt(frame[['open', 'close']].max(axis=1))
                            | frame['low'].gt(frame[['open', 'close']].min(axis=1))
                            | frame['high'].lt(frame['low']))
    flags['negative_volume'] = frame['volume'].lt(0)
    flags['duplicate_date'] = frame.index.duplicated(keep=False)
    return flags

In [ ]:
quality = quality_flags(df)
display(quality.sum())
data_quality_ok = bool(not quality.any().any())
if not data_quality_ok:
    raise ValueError('Data quality checks failed: ' + str(quality.sum().to_dict()))

## 2. From an example to a function
**Retrieve:** for prices 100 → 110 → 99, write the two daily returns and the total return before running this cell.
First we check arithmetic on a tiny Series. Then we put the same operation inside a function so another stock can use it.

In [ ]:
toy_prices = pd.Series([100.0, 110.0, 99.0])
toy_returns = toy_prices / toy_prices.shift(1) - 1
display(toy_returns)
print('Total return:', toy_prices.iloc[-1] / toy_prices.iloc[0] - 1)

In [ ]:
def calculate_returns(prices):
    """Fractional change from the previous observation; the first result is missing."""
    return prices.pct_change(fill_method=None)

df['daily_return'] = calculate_returns(df['adj_close'])
df['log_return'] = np.log(df['adj_close'] / df['adj_close'].shift(1))
df['cumulative_return'] = df['adj_close'] / df['adj_close'].iloc[0] - 1
display(df[['adj_close', 'daily_return', 'cumulative_return']].head())

**Read the function:** `prices` is the input name inside the function; `return` sends its result back to the caller.
`daily_return` and `cumulative_return` answer different questions. The log-return column is provided for homework; its formula is `log(price / previous_price)`.

### Worked example: reuse the return function
For prices 80 → 100 → 90, the daily returns are +25% and −10%. The function returns fractions; multiply by 100 only when displaying percentages.

In [ ]:
example_return_prices = pd.Series([80.0, 100.0, 90.0])
example_returns = calculate_returns(example_return_prices)
display(pd.DataFrame({'price': example_return_prices, 'daily_return': example_returns,
                      'daily_return_pct': example_returns * 100}))

## 3. What does a rolling window contain?
A 3-observation rolling mean has no full-window result until three values are available.
**Predict:** what should the third result be below? Which three prices enter the fourth result?
This same operation will later be used for features in the modeling lessons.

In [ ]:
small = pd.Series([100.0, 110.0, 99.0, 105.0])
display(pd.DataFrame({'price': small, 'mean_3': small.rolling(3).mean()}))

Rolling **volatility** uses the standard deviation of returns. We annualize with the course convention of 252 trading days.
**Predict:** why do 20 returns require 21 prices? Why is a 10-observation estimate more responsive than a 20-observation estimate?
This statistic describes variability, not which stock to buy.

In [ ]:
df['volatility'] = df['daily_return'].rolling(volatility_window).std() * np.sqrt(252)
df['sma_20'] = df['adj_close'].rolling(20).mean()
df['sma_50'] = df['adj_close'].rolling(50).mean()
df['volume_sma_20'] = df['volume'].rolling(20).mean()
df['relative_volume'] = df['volume'] / df['volume_sma_20']
df['daily_range_pct'] = (df['high'] - df['low']) / df['close'] * 100
assert df['volatility'].iloc[:volatility_window].isna().all()
assert np.isfinite(df['volatility'].iloc[volatility_window:]).all()
display(df[['daily_return', 'volatility']].head(volatility_window + 2))

### Drawdown: how far below the previous peak?
For prices `[100, 110, 99, 105]`, running peaks are `[100, 110, 110, 110]`.
Drawdown = price / running peak − 1. The worst drawdown is −10%, even though the final price is above the first.
`cummax()` remembers the largest value seen so far.

In [ ]:
def calculate_drawdown(prices):
    """Fraction below the highest price observed up to each date."""
    return prices / prices.cummax() - 1

df['drawdown'] = calculate_drawdown(df['adj_close'])
display(pd.DataFrame({'price': small, 'running_peak': small.cummax(), 'drawdown': calculate_drawdown(small)}))

## 4. Worked example: worst loss versus final loss
A second path, 80 → 100 → 70 → 85, reaches a −30% drawdown before recovering to −15%. The final price is above its starting value while still below the earlier peak.

In [ ]:
example_drawdown_prices = pd.Series([80.0, 100.0, 70.0, 85.0])
example_drawdown = calculate_drawdown(example_drawdown_prices)
example_worst_drawdown = example_drawdown.min()
display(pd.DataFrame({'price': example_drawdown_prices, 'running_peak': example_drawdown_prices.cummax(),
                      'drawdown': example_drawdown}))
print('Worst:', example_worst_drawdown, '| Final:', example_drawdown.iloc[-1])

### Read the four panels together
The figure uses one Axes object for each panel.
Point to: a price decline, a large daily move, a volatility warm-up gap, and the worst drawdown.
**Change one thing:** add a useful title or label; explain what your change helps a reader understand.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8), layout='constrained')
fig.suptitle(f'{ticker}: {start_date} to {end_date}')
axes[0, 0].plot(df.index, df['adj_close'], label='Adjusted close')
axes[0, 0].plot(df.index, df['sma_20'], label='20-observation mean')
axes[0, 0].set(title='Price and recent average', ylabel='Price'); axes[0, 0].legend()
axes[0, 1].bar(df.index, df['daily_return'] * 100, width=1)
axes[0, 1].set(title='Daily returns', ylabel='Return (%)')
axes[0, 1].axhline(0, color='black', linewidth=0.5)
axes[1, 0].plot(df.index, df['volatility'] * 100, color='purple')
axes[1, 0].set(title=f'{volatility_window}-observation rolling volatility', ylabel='Annualized volatility (%)')
axes[1, 1].fill_between(df.index, df['drawdown'] * 100, 0, alpha=0.4, color='red')
axes[1, 1].set(title='Distance below the previous peak', ylabel='Drawdown (%)')
for ax in axes.flat:
    ax.set_xlabel('Date')
    ax.tick_params(axis='x', labelrotation=20)
plt.show()

## 5. Explain and reuse the result
The summary stores numbers so they can be collected later. `total_return_pct` is in percent; `avg_volatility` is an annualized fraction.
Before reading the table, predict whether the final total return and maximum drawdown must have the same sign.

In [ ]:
# Calculate key statistics
stats = {
    'ticker': ticker,
    'start_date': start_date,
    'end_date': end_date,
    'trading_days': len(df),
    'start_price': df['adj_close'].iloc[0],
    'end_price': df['adj_close'].iloc[-1],
    'total_return_pct': df['cumulative_return'].iloc[-1] * 100,
    'avg_daily_return_pct': df['daily_return'].mean() * 100,
    'daily_return_std_pct': df['daily_return'].std() * 100,
    'avg_volatility': df['volatility'].mean(),
    'max_volatility': df['volatility'].max(),
    'min_volatility': df['volatility'].min(),
    'max_drawdown_pct': df['drawdown'].min() * 100,
    'avg_daily_volume': df['volume'].mean(),
    'max_single_day_return_pct': df['daily_return'].max() * 100,
    'min_single_day_return_pct': df['daily_return'].min() * 100,
}

# Display stats
print(f"\n=== {ticker} Summary: {start_date} to {end_date} ===")
print(f"Trading days: {stats['trading_days']}")
print(f"Price: ${stats['start_price']:.2f} -> ${stats['end_price']:.2f}")
print(f"Total return: {stats['total_return_pct']:.2f}%")
print(f"Average daily return: {stats['avg_daily_return_pct']:.4f}%")
print(f"Daily return std dev: {stats['daily_return_std_pct']:.4f}%")
print(f"Average volatility (annualized): {stats['avg_volatility']:.2%}")
print(f"Max drawdown: {stats['max_drawdown_pct']:.2f}%")
print(f"Best single day: {stats['max_single_day_return_pct']:.2f}%")
print(f"Worst single day: {stats['min_single_day_return_pct']:.2f}%")

### Export named values with Scrapbook
`sb.glue('name', value)` records a result in the executed notebook. The next module retrieves it without recalculating.
The supplied cell records the summary, the underlying price table, and the figure.

In [ ]:
# Glue individual metrics (these can be easily aggregated later)
sb.glue('ticker', ticker)
sb.glue('start_date', start_date)
sb.glue('end_date', end_date)
sb.glue('trading_days', stats['trading_days'])
sb.glue('total_return_pct', stats['total_return_pct'])
sb.glue('avg_volatility', stats['avg_volatility'])
sb.glue('max_drawdown_pct', stats['max_drawdown_pct'])
sb.glue('data_quality_ok', data_quality_ok)

# Glue the full stats dictionary for detailed access
serializable_stats = {key: value.item() if isinstance(value, np.generic) else value for key, value in stats.items()}
sb.glue('stats', serializable_stats)

# Glue the processed dataframe for potential further analysis
sb.glue('prices', df, encoder='pandas')

# Glue the figure
sb.glue('analysis_plot', fig, encoder='display', display=False)

print("Results exported via scrapbook.")
print("These can be collected when running this notebook at scale with papermill.")

Next: [One report becomes three](3_running_analysis_at_scale.ipynb).

## Optional appendix: Yang–Zhang OHLC volatility
The main report uses close-to-close volatility. Yang–Zhang combines sample variances of overnight and open-to-close log returns with a mean Rogers–Satchell term.
The implementation puts OHLC on a consistent adjusted scale before cross-day calculations.
Reference: [Yang and Zhang (2000)](https://papers.ssrn.com/sol3/papers.cfm?abstract_id=229190).

In [ ]:
def yang_zhang_volatility(frame, window=20):
    """Annualized Yang-Zhang estimate using sample variances and mean Rogers-Satchell term."""
    if window < 2:
        raise ValueError('window must be >= 2')
    factor = frame['adj_close'] / frame['close']
    adjusted = frame[['open', 'high', 'low', 'close']].mul(factor, axis=0)
    overnight = np.log(adjusted['open'] / adjusted['close'].shift(1))
    open_close = np.log(adjusted['close'] / adjusted['open'])
    high_open = np.log(adjusted['high'] / adjusted['open'])
    low_open = np.log(adjusted['low'] / adjusted['open'])
    rs = high_open * (high_open - open_close) + low_open * (low_open - open_close)
    k = 0.34 / (1.34 + (window + 1) / (window - 1))
    variance = (overnight.rolling(window).var(ddof=1)
                + k * open_close.rolling(window).var(ddof=1)
                + (1 - k) * rs.rolling(window).mean())
    return np.sqrt(variance.clip(lower=0) * 252)

yz_vol = yang_zhang_volatility(df, volatility_window)
display(pd.DataFrame({'close_to_close': df['volatility'], 'yang_zhang': yz_vol}).tail())

## Exercises
Use the tables and functions above to complete these tasks. Replace the placeholders with your code, then run the checks.

### Your turn F
Call `calculate_returns` on `pd.Series([50.0, 55.0, 44.0])` and assign the result to `your_returns`.
Predict the values first. Explain why the first value should stay missing.
**Stretch:** predict what happens when a missing price appears in the middle; do not silently fill it.

In [ ]:
your_returns = None

In [ ]:
if your_returns is None:
    print('Try F. The two defined returns should be +10% and -20%.')
else:
    assert pd.isna(your_returns.iloc[0])
    np.testing.assert_allclose(your_returns.iloc[1:], [0.1, -0.2])
    print('F passed. The Series stores fractions, not percentage-point numbers.')

### Your turn G: apply drawdown to new data
For `pd.Series([100.0, 120.0, 90.0, 108.0])`:
1. Calculate the drawdown Series as `your_drawdown`.
2. Find its minimum as `your_worst_drawdown`.
3. Explain why the last value and the worst value differ.

**Hint:** use the function, then `.min()`; check at least one row by hand.
**Stretch:** write a two-line function that calculates whole-period return and test it on `[100, 110, 99]`.
Check each of the four rows by hand and relate the values to a drawdown chart.

In [ ]:
your_drawdown = None
your_worst_drawdown = None
your_drawdown_explanation = ''

In [ ]:
if your_drawdown is None or your_worst_drawdown is None:
    print('Complete G and explain the difference between the final and worst drawdown.')
else:
    np.testing.assert_allclose(your_drawdown, [0, 0, -0.25, -0.10], atol=1e-12)
    assert np.isclose(your_worst_drawdown, -0.25)
    assert your_drawdown_explanation.strip()
    print('G passed. Worst drawdown is a path-dependent statistic.')

### Your turn: make one controlled change
Change `ticker` to `MSFT` **or** `volatility_window` to `10`, then Restart & Run All.
Predict which outputs change, find evidence, and explain one result. Restore the defaults afterwards.
A current-day rolling value includes current-day information; later modeling lessons will ask when a feature was actually available.